# Vaani — Cloud Backend (Google Colab)

**Before running this notebook:**
1. Set runtime to GPU: Runtime → Change runtime type → T4 GPU
2. Add your Gemini API key to Colab Secrets (🔑 icon in left sidebar):
   - Name: `GEMINI_API_KEY` · Value: your key · toggle Notebook access ON
3. Your Google Drive should have this structure (already done):
   ```
   My Drive/vaani/
     backend/          ← all .py files
     data/video/avatar_idle.mp4
     known_responses.txt
   ```

Run all cells top to bottom. Cell 8 prints your public URL — paste it into `frontend/.env.local`.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version:   {torch.version.cuda}")
print(f"GPU:            {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — change runtime type to T4 GPU'}")

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
from pathlib import Path

DRIVE_VAANI   = Path('/content/drive/MyDrive/vaani')
DRIVE_BACKEND = DRIVE_VAANI / 'backend'
DRIVE_DATA    = DRIVE_VAANI / 'data'
DRIVE_MODELS  = DRIVE_VAANI / 'models'

# Show what's actually in Drive so path issues are obvious
print("Contents of My Drive/vaani/:")
if DRIVE_VAANI.exists():
    for p in sorted(DRIVE_VAANI.iterdir()):
        print(f"  {'[dir] ' if p.is_dir() else '[file]'} {p.name}")
else:
    print("  !! vaani/ not found — check spelling in Drive")

DRIVE_MODELS.mkdir(parents=True, exist_ok=True)

assert DRIVE_BACKEND.exists(), \
    f"backend/ not found at {DRIVE_BACKEND}"
assert (DRIVE_DATA / 'video' / 'avatar_idle.mp4').exists(), \
    f"avatar_idle.mp4 not found at {DRIVE_DATA / 'video' / 'avatar_idle.mp4'}"

print("\nAll paths verified. Ready to continue.")

In [ ]:
# ── Cell 3: Install Python dependencies ───────────────────────────────
# Versions chosen to match Colab's pre-installed ecosystem (gradio, google-adk, etc.)
# to avoid resolver conflicts.
!pip install -q \
  'fastapi>=0.115.2' \
  'uvicorn[standard]>=0.30.0' \
  'google-genai>=1.64.0' \
  'httpx>=0.27.1' \
  'websockets>=15.0.1' \
  python-dotenv \
  librosa \
  pyyaml \
  psycopg2-binary \
  supabase

# diffusers pinned — MuseTalk was built against 0.30.x API
!pip install -q \
  opencv-python-headless \
  'diffusers==0.30.2' \
  accelerate \
  omegaconf \
  transformers \
  huggingface_hub \
  timm einops onnxruntime-gpu face-alignment

# Verify critical imports
import numpy, cv2, torch, fastapi, google.genai
print(f"numpy        {numpy.__version__}")
print(f"torch        {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
print(f"fastapi      {fastapi.__version__}")
print(f"google-genai {google.genai.__version__}")
print("All dependencies OK.")

In [ ]:
# ── Cell 4: Clone MuseTalk ────────────────────────────────────────────
if not Path('/content/musetalk').exists():
    !git clone --depth 1 https://github.com/TMElyralab/MuseTalk.git /content/musetalk
else:
    print("MuseTalk already cloned.")

# Symlink so MuseTalk's internal model lookups hit the Drive models dir
musetalk_models = Path('/content/musetalk/models')
if musetalk_models.exists() and not musetalk_models.is_symlink():
    import shutil
    shutil.rmtree(str(musetalk_models))
if not musetalk_models.is_symlink():
    os.symlink(str(DRIVE_MODELS), str(musetalk_models))

print("MuseTalk ready at /content/musetalk")

In [ ]:
# ── Cell 5: Download model weights (skips anything already in Drive) ──
# First run: ~15 min, ~5 GB written to Drive.
# Every run after that: instant — files are already in Drive.
import shutil
from huggingface_hub import snapshot_download, hf_hub_download

def need_download(path: Path) -> bool:
    return not path.exists() or not any(path.iterdir())

def copy_if_missing(src: Path, dst: Path):
    dst.mkdir(parents=True, exist_ok=True)
    if src.exists():
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)

# Pull full MuseTalk HF repo once into Drive cache
hf_cache = DRIVE_MODELS / '_hf_cache'
if need_download(hf_cache):
    print("[1/4] Downloading MuseTalk from HuggingFace (~3 GB)...")
    snapshot_download(
        repo_id='TMElyralab/MuseTalk',
        local_dir=str(hf_cache),
        ignore_patterns=['*.git*', '*.gitattributes'],
    )
else:
    print("[1/4] MuseTalk HF cache already in Drive — skipping.")

# musetalkV15 UNet weights
copy_if_missing(hf_cache / 'models' / 'musetalkV15', DRIVE_MODELS / 'musetalkV15')
print("[2/4] musetalkV15 weights ready.")

# SD-VAE
vae_dst = DRIVE_MODELS / 'sd-vae'
if need_download(vae_dst):
    copied = False
    for candidate in ['sd-vae-ft-mse', 'sd-vae']:
        src = hf_cache / 'models' / candidate
        if src.exists():
            copy_if_missing(src, vae_dst)
            copied = True
            break
    if not copied:
        print("  Downloading SD-VAE directly...")
        snapshot_download('stabilityai/sd-vae-ft-mse', local_dir=str(vae_dst))
print("[3/4] SD-VAE ready.")

# Whisper
whisper_dst = DRIVE_MODELS / 'whisper'
if need_download(whisper_dst):
    src = hf_cache / 'models' / 'whisper'
    if src.exists():
        copy_if_missing(src, whisper_dst)
    else:
        print("  Downloading Whisper tiny...")
        snapshot_download('openai/whisper-tiny', local_dir=str(whisper_dst))
print("[4/4] Whisper ready.")

# DWPose ONNX (face landmark detection)
dwpose_dst = DRIVE_MODELS / 'dwpose'
if need_download(dwpose_dst):
    src = hf_cache / 'models' / 'dwpose'
    if src.exists():
        copy_if_missing(src, dwpose_dst)
    else:
        print("  Downloading DWPose ONNX...")
        dwpose_dst.mkdir(parents=True, exist_ok=True)
        for fname in ['dw-ll_ucoco_384.onnx', 'yolox_l.onnx']:
            try:
                hf_hub_download(repo_id='yzd-v/DWPose', filename=fname, local_dir=str(dwpose_dst))
            except Exception as e:
                print(f"  Warning: {fname}: {e}")
print("DWPose ready.")

print("\nAll models ready.")

In [ ]:
# ── Cell 5b: Fix model paths ──────────────────────────────────────────
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

DRIVE_MODELS = Path('/content/drive/MyDrive/vaani/models')

# Copy unet weights to where config.py expects them
dst = DRIVE_MODELS / 'musetalkV15'
dst.mkdir(exist_ok=True)
shutil.copy2(DRIVE_MODELS / '_hf_cache/musetalkV15/unet.pth',     dst / 'unet.pth')
shutil.copy2(DRIVE_MODELS / '_hf_cache/musetalkV15/musetalk.json', dst / 'musetalk.json')
print(f"unet.pth      {(dst / 'unet.pth').stat().st_size // 1024 // 1024} MB  OK")
print(f"musetalk.json OK")

# Download missing yolox_l.onnx for DWPose face detection
dwpose_dst = DRIVE_MODELS / 'dwpose'
try:
    hf_hub_download(repo_id='yzd-v/DWPose', filename='yolox_l.onnx', local_dir=str(dwpose_dst))
    print("yolox_l.onnx  OK")
except Exception as e:
    print(f"yolox_l.onnx  failed ({e}) — will fall back to face-alignment library")

print("\nAll done — re-run Cell 7 now.")

In [ ]:
# ── Cell 6: Set up paths and environment ──────────────────────────────
import sys
from google.colab import userdata

# Python path
sys.path.insert(0, str(DRIVE_BACKEND))   # backend .py files
sys.path.insert(0, '/content/musetalk')  # MuseTalk source

# Prerendered dir (may be empty — that's fine)
(DRIVE_VAANI / 'prerendered').mkdir(parents=True, exist_ok=True)

# config.py path resolution:
#   BASE_DIR = Path(DRIVE_BACKEND / 'config.py').parent.parent = DRIVE_VAANI
#   MUSETALK_DIR  → overridden by env var to /content/musetalk
#   models/       → DRIVE_VAANI/models  (already exists)
#   data/         → DRIVE_VAANI/data    (already exists)
#   prerendered/  → DRIVE_VAANI/prerendered
#   known_responses.txt → DRIVE_VAANI/known_responses.txt

os.environ['GEMINI_API_KEY']    = userdata.get('GEMINI_API_KEY')
os.environ['AGENT_NAME']        = 'Vaani'
os.environ['AGENT_ROLE']        = 'Your AI Assistant'
os.environ['AGENT_VOICE']       = 'Puck'
os.environ['MUSETALK_ENABLED']  = 'true'
os.environ['MUSETALK_BATCH_MS'] = '200'
os.environ['MUSETALK_DIR']      = '/content/musetalk'
os.environ['CORS_ORIGINS']      = '*'
os.environ['BACKEND_PORT']      = '8000'
os.environ['MPLBACKEND']        = 'Agg'

print("Environment configured.")
print(f"  GEMINI_API_KEY : {'set' if os.environ.get('GEMINI_API_KEY') else 'MISSING — add to Colab Secrets!'}")
print(f"  MUSETALK_DIR   : {os.environ['MUSETALK_DIR']}")
print(f"  BASE_DIR       : {DRIVE_VAANI}")

In [ ]:
# ── Cell 7: Start the FastAPI server ──────────────────────────────────
import subprocess
import threading
import time

server_log = []

def run_server():
    proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'main:app',
         '--host', '0.0.0.0', '--port', '8000', '--log-level', 'info'],
        cwd=str(DRIVE_BACKEND),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    for line in proc.stdout:
        text = line.decode('utf-8', errors='replace').rstrip()
        server_log.append(text)
        print(f'[SERVER] {text}')

threading.Thread(target=run_server, daemon=True).start()

print("Waiting for server (MuseTalk model load ~30 sec)...")
for _ in range(120):
    time.sleep(1)
    if any('Ready. Waiting for connections' in l for l in server_log):
        print("\nServer is ready!")
        break
    if any('GEMINI_API_KEY is not set' in l for l in server_log):
        print("\nMissing GEMINI_API_KEY — add it to Colab Secrets and re-run Cell 6.")
        break
else:
    print("\nStill loading — continue to Cell 8 anyway.")

In [ ]:
# ── Cell 8: Open public tunnel via Cloudflare ─────────────────────────
import re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
     -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

tunnel_log = []

def run_tunnel():
    proc = subprocess.Popen(
        ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in proc.stdout:
        tunnel_log.append(line.decode('utf-8', errors='replace').rstrip())

threading.Thread(target=run_tunnel, daemon=True).start()

public_url = None
for _ in range(30):
    time.sleep(1)
    for line in tunnel_log:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group()
            break
    if public_url:
        break

if public_url:
    print('=' * 60)
    print(f'  Backend URL: {public_url}')
    print()
    print('  Paste into frontend/.env.local on your MacBook:')
    print(f'  VITE_BACKEND_URL={public_url}')
    print('=' * 60)
    print()
    print('  Then run:  cd frontend && npm run dev')
else:
    print('URL not found yet. Tunnel logs:')
    for l in tunnel_log: print(l)